# 🧠 Bag of Words (BoW) — Practical Implementation

## 📌 What is Bag of Words?

**Bag of Words (BoW)** is a text vectorization technique used to convert textual data into numerical features that machine learning algorithms can understand.

Machine learning models cannot directly process raw text such as:

`Congratulations! You have won a free prize.`

Therefore, the text must first be converted into a numerical representation.

Bag of Words represents each document based on the occurrence or frequency of words from a vocabulary.

---

## 🔄 Complete NLP Pipeline

The practical implementation follows this pipeline:

**Raw SMS Messages**

⬇️

**Text Cleaning**

⬇️

**Lowercasing**

⬇️

**Tokenization**

⬇️

**Stopword Removal**

⬇️

**Word Normalization**

**Stemming / Lemmatization**

⬇️

**Bag of Words Vectorization**

⬇️

**Numerical Feature Matrix**

⬇️

**Machine Learning Model**

---

## 🎯 Objective

In this notebook, we will:

1. Load the **SMS Spam Collection dataset**.
2. Clean and preprocess the text.
3. Apply **Snowball Stemming**.
4. Apply **WordNet Lemmatization** as an alternative preprocessing technique.
5. Convert text into numerical vectors using **Bag of Words**.
6. Understand the generated vocabulary.
7. Compare **Count BoW** and **Binary BoW**.
8. Explore **N-grams** using `CountVectorizer`.

> Stemming and lemmatization will be implemented as separate preprocessing approaches so that their results can be compared independently.

In [1]:
import re
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer

In [2]:
# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/milind/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/milind/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/milind/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# 📂 1. Load the Dataset

The dataset contains SMS messages classified into two categories:

| Label | Meaning |
|---|---|
| `ham` | Normal / legitimate message |
| `spam` | Unwanted or promotional message |

The dataset contains two columns:

- **label** — the target class
- **message** — the actual SMS text

In [3]:
# Load the SMS Spam Collection dataset
messages = pd.read_csv(
    "SMSSpamCollection.txt",
    sep="\t",
    names=["label", "message"]
)

# Display the first five records
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
print("Dataset Shape:", messages.shape)

messages.info()

Dataset Shape: (5572, 2)
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 542.9 KB


In [5]:
messages["label"].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

# 🧹 2. Text Cleaning and Preprocessing

Raw text usually contains information that may not be useful for a basic Bag of Words model.

The preprocessing steps used in this notebook are:

1. Remove non-alphabetic characters.
2. Convert text to lowercase.
3. Split the text into individual words.
4. Remove English stopwords.
5. Apply word normalization.
6. Join the processed words back into a sentence.

We will create two separate versions of the corpus:

### Corpus 1 — Snowball Stemming

`Raw Text → Cleaning → Lowercase → Stopword Removal → Snowball Stemming`

### Corpus 2 — WordNet Lemmatization

`Raw Text → Cleaning → Lowercase → Stopword Removal → Lemmatization`

This allows us to compare the effect of both normalization techniques.

In [6]:
# Store English stopwords in a set for efficient lookup
english_stopwords = set(stopwords.words("english"))

# Initialize Snowball Stemmer
snowball_stemmer = SnowballStemmer("english")

# Initialize WordNet Lemmatizer
lemmatizer = WordNetLemmatizer()

# ❄️ 3. Text Preprocessing Using Snowball Stemmer

**SnowballStemmer** reduces words to their root-like forms by removing prefixes and suffixes according to predefined linguistic rules.

For example:

| Original Word | Stem |
|---|---|
| `playing` | `play` |
| `played` | `play` |
| `studies` | `studi` |
| `connection` | `connect` |

A stem is **not guaranteed to be a valid dictionary word**.

For example:

`studies → studi`

This is expected because stemming focuses on reducing related words to a common representation rather than producing grammatically valid words.

In [7]:
def preprocess_with_stemming(text):
    """
    Clean and preprocess text using Snowball stemming.

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    str
        Cleaned and stemmed text.
    """

    # Step 1: Remove non-alphabetic characters
    text = re.sub(r"[^a-zA-Z]", " ", text)

    # Step 2: Convert text to lowercase
    text = text.lower()

    # Step 3: Split text into individual words
    words = text.split()

    # Step 4: Remove stopwords and apply Snowball stemming
    processed_words = [
        snowball_stemmer.stem(word)
        for word in words
        if word not in english_stopwords
    ]

    # Step 5: Join processed words back into a sentence
    return " ".join(processed_words)

In [8]:
# Apply stemming preprocessing to every SMS message
stemmed_corpus = [
    preprocess_with_stemming(message)
    for message in messages["message"]
]

In [9]:
print("Original Message:")
print(messages["message"].iloc[0])

print("\nAfter Snowball Stemming:")
print(stemmed_corpus[0])

Original Message:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

After Snowball Stemming:
go jurong point crazi avail bugi n great world la e buffet cine got amor wat


# 📖 4. Text Preprocessing Using WordNet Lemmatizer

**Lemmatization** reduces words to meaningful dictionary forms called **lemmas**.

Unlike stemming, lemmatization attempts to produce linguistically valid base words.

For example:

| Original Word | Lemma |
|---|---|
| `cars` | `car` |
| `messages` | `message` |
| `children` | `child` |
| `studies` | `study` |

In this basic Bag of Words implementation, words are lemmatized using the default noun POS.

This approach is computationally simpler, but it does not correctly normalize every verb or adjective.

For more advanced preprocessing, **POS-aware lemmatization** can be used:

`Tokenization → POS Tagging → POS Mapping → Lemmatization`

However, for understanding the core Bag of Words pipeline, basic WordNet lemmatization provides a clean starting point.

In [10]:
def preprocess_with_lemmatization(text):
    """
    Clean and preprocess text using WordNet lemmatization.

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    str
        Cleaned and lemmatized text.
    """

    # Step 1: Remove non-alphabetic characters
    text = re.sub(r"[^a-zA-Z]", " ", text)

    # Step 2: Convert text to lowercase
    text = text.lower()

    # Step 3: Split text into individual words
    words = text.split()

    # Step 4: Remove stopwords and apply lemmatization
    processed_words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in english_stopwords
    ]

    # Step 5: Join processed words back into a sentence
    return " ".join(processed_words)

In [11]:
# Apply lemmatization preprocessing to every SMS message
lemmatized_corpus = [
    preprocess_with_lemmatization(message)
    for message in messages["message"]
]

In [12]:
print("Original Message:")
print(messages["message"].iloc[0])

print("\nSnowball Stemmed:")
print(stemmed_corpus[0])

print("\nLemmatized:")
print(lemmatized_corpus[0])

Original Message:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Snowball Stemmed:
go jurong point crazi avail bugi n great world la e buffet cine got amor wat

Lemmatized:
go jurong point crazy available bugis n great world la e buffet cine got amore wat


# 🔍 5. Stemming vs Lemmatization

Both techniques attempt to normalize words, but they work differently.

| Feature | Snowball Stemming | WordNet Lemmatization |
|---|---|---|
| Approach | Rule-based suffix removal | Dictionary-based normalization |
| Output | May not be a valid word | Usually a meaningful word |
| Speed | Faster | Slightly slower |
| Linguistic Accuracy | Lower | Higher |
| Example | `studies → studi` | `studies → study` |

Neither technique is universally better.

For traditional text classification tasks such as spam detection, stemming can sometimes work well because the goal is often to group similar word forms rather than preserve perfect linguistic meaning.

The best approach should ultimately be selected through model evaluation.

# 🔢 6. Creating the Bag of Words Model

After preprocessing, the text still consists of words.

Machine learning algorithms require numerical input.

`CountVectorizer` converts the processed text into a numerical **Document-Term Matrix**.

Suppose our vocabulary is:

`["free", "prize", "win"]`

Then:

`"free prize" → [1, 1, 0]`

`"win free prize" → [1, 1, 1]`

Each:

- **Row** represents one document or SMS message.
- **Column** represents one vocabulary word.
- **Value** represents the frequency of that word in the document.

In [13]:
# Create the Bag of Words vectorizer
stemmed_vectorizer = CountVectorizer(
    max_features=100
)

# Learn the vocabulary and transform the stemmed corpus
X_stemmed = stemmed_vectorizer.fit_transform(stemmed_corpus)

print("BoW Matrix Shape:", X_stemmed.shape)

BoW Matrix Shape: (5572, 100)


In [14]:
X_stemmed[:5].toarray()

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0,

In [15]:
np.X_stemmed.value_counts()

AttributeError: 'csr_matrix' object has no attribute 'value_counts'

In [17]:
# View the selected vocabulary:
stemmed_vectorizer.get_feature_names_out()

array(['alreadi', 'amp', 'ask', 'babe', 'back', 'call', 'care', 'cash',
       'claim', 'come', 'da', 'day', 'dear', 'dont', 'even', 'feel',
       'find', 'free', 'friend', 'get', 'give', 'go', 'good', 'got',
       'great', 'gt', 'happi', 'hey', 'hi', 'home', 'hope', 'im', 'keep',
       'know', 'last', 'later', 'leav', 'let', 'life', 'like', 'lor',
       'love', 'lt', 'make', 'meet', 'messag', 'min', 'miss', 'mobil',
       'morn', 'msg', 'much', 'need', 'new', 'night', 'number', 'oh',
       'ok', 'one', 'phone', 'pick', 'pleas', 'pls', 'prize', 'realli',
       'repli', 'right', 'said', 'say', 'see', 'send', 'sleep', 'sorri',
       'still', 'stop', 'take', 'tell', 'text', 'thank', 'thing', 'think',
       'time', 'today', 'tomorrow', 'tone', 'tri', 'txt', 'ur', 'wait',
       'want', 'wat', 'way', 'week', 'well', 'win', 'work', 'www', 'yeah',
       'year', 'yes'], dtype=object)

## BoW with Lemmatization

In [18]:
# Create a separate vectorizer for the lemmatized corpus
lemma_vectorizer = CountVectorizer(
    max_features=100
)

# Create Bag of Words features
X_lemmatized = lemma_vectorizer.fit_transform(
    lemmatized_corpus
)

print("BoW Matrix Shape:", X_lemmatized.shape)

BoW Matrix Shape: (5572, 100)


In [19]:
lemma_vectorizer.get_feature_names_out()

array(['already', 'amp', 'anything', 'ask', 'babe', 'back', 'call',
       'care', 'cash', 'claim', 'co', 'com', 'come', 'da', 'day', 'dear',
       'dont', 'find', 'free', 'friend', 'get', 'give', 'go', 'going',
       'good', 'got', 'great', 'gt', 'happy', 'hey', 'hi', 'home', 'hope',
       'im', 'know', 'last', 'later', 'let', 'life', 'like', 'lor',
       'love', 'lt', 'make', 'meet', 'message', 'min', 'miss', 'mobile',
       'morning', 'msg', 'much', 'need', 'new', 'night', 'nokia',
       'number', 'oh', 'ok', 'one', 'phone', 'please', 'pls', 'prize',
       'really', 'reply', 'right', 'said', 'say', 'see', 'send',
       'service', 'sorry', 'still', 'stop', 'take', 'tell', 'text',
       'thanks', 'thing', 'think', 'time', 'today', 'tomorrow', 'tone',
       'txt', 'uk', 'ur', 'want', 'wat', 'way', 'week', 'well', 'win',
       'work', 'would', 'www', 'yeah', 'year', 'yes'], dtype=object)

# 7️⃣ Count-Based BoW vs Binary BoW

`CountVectorizer` can represent words in two main ways.

## Count-Based Bag of Words

With:

`binary=False`

the value represents the **number of times** a word appears.

For example:

`free free prize`

may be represented as:

`[2, 1]`

## Binary Bag of Words

With:

`binary=True`

the vector only represents whether a word is present.

The same text:

`free free prize`

becomes:

`[1, 1]`

Therefore:

| Representation | Meaning |
|---|---|
| `0` | Word is absent |
| `1` | Word is present |

Repeated occurrences do not increase the value in Binary BoW.

Binary BoW can be useful when the **presence of a word** is more important than its frequency.

## Binary BoW

In [20]:
binary_vectorizer = CountVectorizer(
    max_features=100,
    binary=True
)

X_binary = binary_vectorizer.fit_transform(
    stemmed_corpus
)

print("Binary BoW Shape:", X_binary.shape)

Binary BoW Shape: (5572, 100)


In [21]:
X_binary[:5].toarray()

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0,

# 📚 8. Understanding the Vocabulary

During `fit_transform()`, `CountVectorizer` learns a vocabulary from the training corpus.

Each unique selected word receives a column index.

For example:

`free → 0`

`message → 1`

`prize → 2`

The vocabulary determines the meaning of each column in the Bag of Words matrix.

In [22]:
# Display word-to-column mappings
stemmed_vectorizer.vocabulary_

{'go': np.int64(21),
 'great': np.int64(24),
 'got': np.int64(23),
 'wat': np.int64(90),
 'ok': np.int64(57),
 'free': np.int64(17),
 'win': np.int64(94),
 'text': np.int64(77),
 'txt': np.int64(86),
 'say': np.int64(68),
 'alreadi': np.int64(0),
 'think': np.int64(80),
 'hey': np.int64(27),
 'week': np.int64(92),
 'back': np.int64(4),
 'like': np.int64(39),
 'still': np.int64(73),
 'send': np.int64(70),
 'even': np.int64(14),
 'friend': np.int64(18),
 'prize': np.int64(63),
 'claim': np.int64(8),
 'call': np.int64(5),
 'mobil': np.int64(48),
 'home': np.int64(29),
 'want': np.int64(89),
 'today': np.int64(82),
 'cash': np.int64(7),
 'day': np.int64(11),
 'repli': np.int64(65),
 'www': np.int64(96),
 'right': np.int64(66),
 'thank': np.int64(78),
 'take': np.int64(75),
 'time': np.int64(81),
 'messag': np.int64(45),
 'oh': np.int64(56),
 'yes': np.int64(99),
 'make': np.int64(43),
 'way': np.int64(91),
 'feel': np.int64(15),
 'dont': np.int64(13),
 'miss': np.int64(47),
 'ur': np.int64

In [23]:
# Display vocabulary in column order
for index, word in enumerate(
    stemmed_vectorizer.get_feature_names_out()
):
    print(f"{index:<5} ---> {word}")

0     ---> alreadi
1     ---> amp
2     ---> ask
3     ---> babe
4     ---> back
5     ---> call
6     ---> care
7     ---> cash
8     ---> claim
9     ---> come
10    ---> da
11    ---> day
12    ---> dear
13    ---> dont
14    ---> even
15    ---> feel
16    ---> find
17    ---> free
18    ---> friend
19    ---> get
20    ---> give
21    ---> go
22    ---> good
23    ---> got
24    ---> great
25    ---> gt
26    ---> happi
27    ---> hey
28    ---> hi
29    ---> home
30    ---> hope
31    ---> im
32    ---> keep
33    ---> know
34    ---> last
35    ---> later
36    ---> leav
37    ---> let
38    ---> life
39    ---> like
40    ---> lor
41    ---> love
42    ---> lt
43    ---> make
44    ---> meet
45    ---> messag
46    ---> min
47    ---> miss
48    ---> mobil
49    ---> morn
50    ---> msg
51    ---> much
52    ---> need
53    ---> new
54    ---> night
55    ---> number
56    ---> oh
57    ---> ok
58    ---> one
59    ---> phone
60    ---> pick
61    ---> pleas
62    ---> pls
63  

# 🔗 9. N-Grams in Bag of Words

A standard Bag of Words model usually treats individual words independently.

These individual words are called **unigrams**.

For example:

`machine learning is powerful`

### Unigrams

`machine`

`learning`

`powerful`

However, individual words may lose important contextual information.

N-grams allow Bag of Words to represent sequences of consecutive words.

## Bigrams

Two consecutive words:

`machine learning`

`learning powerful`

## Trigrams

Three consecutive words:

`machine learning powerful`

The `ngram_range` parameter controls which n-grams are generated.

| Setting | Meaning |
|---|---|
| `(1, 1)` | Unigrams only |
| `(2, 2)` | Bigrams only |
| `(3, 3)` | Trigrams only |
| `(1, 2)` | Unigrams + Bigrams |
| `(1, 3)` | Unigrams + Bigrams + Trigrams |
| `(2, 3)` | Bigrams + Trigrams |

Using n-grams can capture more contextual information, but it also increases the number of features.

In [24]:
ngram_vectorizer = CountVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    binary=True
)

X_ngrams = ngram_vectorizer.fit_transform(
    stemmed_corpus
)

print("N-Gram Matrix Shape:", X_ngrams.shape)

N-Gram Matrix Shape: (5572, 500)


In [25]:
ngram_vectorizer.get_feature_names_out()[:50]

array(['abl', 'abt', 'account', 'actual', 'address', 'afternoon', 'age',
       'ah', 'aight', 'alreadi', 'alright', 'also', 'alway', 'amp',
       'anoth', 'answer', 'anyth', 'anyway', 'appli', 'ard', 'around',
       'ask', 'attempt', 'await', 'await collect', 'award', 'away',
       'babe', 'babi', 'back', 'bad', 'bath', 'beauti', 'bed', 'believ',
       'best', 'better', 'big', 'birthday', 'bit', 'bonus', 'book',
       'bore', 'box', 'boy', 'bring', 'brother', 'bt', 'bus', 'busi'],
      dtype=object)

In [26]:
bigram_trigram_vectorizer = CountVectorizer(
    max_features=500,
    ngram_range=(2, 3),
    binary=True
)

X_bigram_trigram = bigram_trigram_vectorizer.fit_transform(
    stemmed_corpus
)

print(
    "Bigram + Trigram Matrix Shape:",
    X_bigram_trigram.shape
)

Bigram + Trigram Matrix Shape: (5572, 500)


# 📊 10. Understanding the Final Feature Matrix

After applying Bag of Words:

`X.shape = (Number of Documents, Number of Features)`

For example:

`(5572, 100)`

means:

- **5572 rows** → 5572 SMS messages
- **100 columns** → 100 selected vocabulary features

Each row represents one SMS message as a numerical vector.

Each column represents a word or n-gram from the learned vocabulary.

---

## ⚠️ Important: Sparse vs Dense Matrix

`CountVectorizer` returns a **sparse matrix** by default.

This is memory-efficient because most Bag of Words values are `0`.

Therefore, for practical machine learning:

`X = vectorizer.fit_transform(corpus)`

is preferred over converting the complete matrix using:

`X = vectorizer.fit_transform(corpus).toarray()`

Use `.toarray()` only when inspecting small portions of the matrix during learning.

For example:

`X[:5].toarray()`

This avoids unnecessary memory consumption when working with larger datasets.

# 🧠 Key Takeaways

- **Bag of Words** converts textual documents into numerical feature vectors.
- Text should usually be cleaned and normalized before vectorization.
- **SnowballStemmer** reduces related words to common stem forms.
- **WordNetLemmatizer** attempts to produce meaningful dictionary base forms.
- Stemming and lemmatization should be tested as **separate preprocessing strategies**.
- `CountVectorizer` automatically learns the vocabulary from the corpus.
- `max_features` limits the number of vocabulary features.
- `binary=True` records word presence instead of word frequency.
- `ngram_range` allows the model to capture sequences of words.
- Bag of Words creates a **Document-Term Matrix**.
- Sparse matrices should generally be preserved for memory efficiency.
- The choice between stemming and lemmatization should ultimately be validated using downstream model performance.

---

# 🏆 Complete Bag of Words Pipeline

## Using Snowball Stemming

**Raw SMS**

⬇️

**Text Cleaning**

⬇️

**Lowercasing**

⬇️

**Stopword Removal**

⬇️

**Snowball Stemming**

⬇️

**CountVectorizer**

⬇️

**Bag of Words Matrix**

⬇️

**Machine Learning Model**

---

## Using Lemmatization

**Raw SMS**

⬇️

**Text Cleaning**

⬇️

**Lowercasing**

⬇️

**Stopword Removal**

⬇️

**WordNet Lemmatization**

⬇️

**CountVectorizer**

⬇️

**Bag of Words Matrix**

⬇️

**Machine Learning Model**

---

## 📌 In Short

`Raw Text → Preprocessing → Word Normalization → Bag of Words → Numerical Features`

# 🧠 POS-Aware Lemmatization for Bag of Words

## 📌 Why Use POS-Aware Lemmatization?

The `WordNetLemmatizer` produces more accurate lemmas when the correct **Part-of-Speech (POS)** information is provided.

By default, `WordNetLemmatizer` treats a word as a **noun**.

For example:

| Word | Default Lemmatization | Correct POS | POS-Aware Lemma |
|---|---|---|---|
| `cars` | `car` | Noun | `car` |
| `running` | `running` | Verb | `run` |
| `went` | `went` | Verb | `go` |
| `better` | `better` | Adjective | `good` |
| `children` | `child` | Noun | `child` |

Therefore, basic lemmatization may fail to reduce verbs, adjectives, and adverbs correctly.

POS-aware lemmatization solves this problem by first identifying the grammatical role of each word and then passing the appropriate POS information to `WordNetLemmatizer`.

---

# 🔄 POS-Aware Lemmatization Pipeline

The complete preprocessing pipeline is:

**Raw SMS Message**

⬇️

**Text Cleaning**

⬇️

**Lowercasing**

⬇️

**Word Tokenization**

⬇️

**POS Tagging**

⬇️

**Convert NLTK POS Tags → WordNet POS Tags**

⬇️

**Stopword Removal**

⬇️

**POS-Aware Lemmatization**

⬇️

**Processed Corpus**

⬇️

**CountVectorizer**

⬇️

**Bag of Words Matrix**

This approach produces a more linguistically informed text representation before creating the Bag of Words features.

In [27]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import wordnet

In [28]:
# Download required NLTK resources
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /Users/milind/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/milind/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/milind/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /Users/milind/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/milind/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# 🔄 Converting NLTK POS Tags to WordNet POS Tags

NLTK's `pos_tag()` and `WordNetLemmatizer` use different POS-tagging systems.

For example, `pos_tag()` may return:

`running → VBG`

However, `WordNetLemmatizer` expects:

`wordnet.VERB`

Therefore, the Penn Treebank POS tags returned by NLTK must be mapped to their corresponding WordNet POS categories.

| NLTK Tag Starts With | Word Type | WordNet POS |
|---|---|---|
| `J` | Adjective | `wordnet.ADJ` |
| `V` | Verb | `wordnet.VERB` |
| `N` | Noun | `wordnet.NOUN` |
| `R` | Adverb | `wordnet.ADV` |

Examples:

`VBG → Verb → wordnet.VERB`

`NNS → Noun → wordnet.NOUN`

`JJR → Adjective → wordnet.ADJ`

`RB → Adverb → wordnet.ADV`

WordNetLemmatizer mainly supports these four grammatical categories for lemmatization.

Words belonging to unsupported POS categories can simply be preserved without lemmatization.

In [29]:
def get_wordnet_pos(tag):
    """
    Convert a Penn Treebank POS tag into a WordNet POS tag.

    Parameters
    ----------
    tag : str
        POS tag generated by NLTK's pos_tag().

    Returns
    -------
    str or None
        Corresponding WordNet POS tag.
        Returns None when the POS category is unsupported.
    """

    if tag.startswith("J"):
        return wordnet.ADJ

    elif tag.startswith("V"):
        return wordnet.VERB

    elif tag.startswith("N"):
        return wordnet.NOUN

    elif tag.startswith("R"):
        return wordnet.ADV

    else:
        return None

# 🧹 POS-Aware Text Preprocessing

The following function performs the complete preprocessing operation for a single SMS message.

The steps are:

1. Remove non-alphabetic characters.
2. Convert the text to lowercase.
3. Tokenize the text into individual words.
4. Remove English stopwords.
5. Assign POS tags to the remaining words.
6. Convert NLTK POS tags into WordNet POS tags.
7. Lemmatize each supported word using its detected POS.
8. Preserve unsupported words without lemmatization.
9. Join the processed words back into a single string.

The resulting text can then be passed directly to `CountVectorizer`.

In [30]:
def preprocess_with_pos_lemmatization(text):
    """
    Clean and preprocess text using POS-aware WordNet lemmatization.

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    str
        Cleaned and POS-aware lemmatized text.
    """

    # Step 1: Remove non-alphabetic characters
    text = re.sub(r"[^a-zA-Z]", " ", text)

    # Step 2: Convert text to lowercase
    text = text.lower()

    # Step 3: Tokenize the cleaned text
    words = word_tokenize(text)

    # Step 4: Remove English stopwords
    filtered_words = [
        word
        for word in words
        if word not in english_stopwords
    ]

    # Step 5: Assign POS tags
    tagged_words = pos_tag(filtered_words)

    # Store the final processed words
    lemmatized_words = []

    # Step 6: Perform POS-aware lemmatization
    for word, tag in tagged_words:

        # Convert NLTK POS tag to WordNet POS
        wordnet_pos = get_wordnet_pos(tag)

        if wordnet_pos:
            # Lemmatize using the detected POS
            lemma = lemmatizer.lemmatize(
                word,
                pos=wordnet_pos
            )

        else:
            # Keep unsupported POS categories unchanged
            lemma = word

        lemmatized_words.append(lemma)

    # Step 7: Reconstruct the processed text
    return " ".join(lemmatized_words)

# 📚 Creating the POS-Aware Lemmatized Corpus

The preprocessing function is now applied independently to every SMS message in the dataset.

Each original SMS message is converted into a cleaned and linguistically normalized version.

The original `messages` DataFrame remains unchanged.

In [31]:
# Apply POS-aware lemmatization to every SMS message
pos_lemmatized_corpus = [
    preprocess_with_pos_lemmatization(message)
    for message in messages["message"]
]

print(
    "Total Processed Messages:",
    len(pos_lemmatized_corpus)
)

Total Processed Messages: 5572


In [32]:
# Select a message for comparison
index = 0

print("ORIGINAL MESSAGE")
print("-" * 70)
print(messages["message"].iloc[index])


print("\nSNOWBALL STEMMING")
print("-" * 70)
print(stemmed_corpus[index])


print("\nBASIC LEMMATIZATION")
print("-" * 70)
print(lemmatized_corpus[index])


print("\nPOS-AWARE LEMMATIZATION")
print("-" * 70)
print(pos_lemmatized_corpus[index])

ORIGINAL MESSAGE
----------------------------------------------------------------------
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

SNOWBALL STEMMING
----------------------------------------------------------------------
go jurong point crazi avail bugi n great world la e buffet cine got amor wat

BASIC LEMMATIZATION
----------------------------------------------------------------------
go jurong point crazy available bugis n great world la e buffet cine got amore wat

POS-AWARE LEMMATIZATION
----------------------------------------------------------------------
go jurong point crazy available bugis n great world la e buffet cine get amore wat


In [33]:
sample_text = (
    "The children were running quickly and eating "
    "better foods while their parents went shopping."
)

processed_text = preprocess_with_pos_lemmatization(
    sample_text
)

print("Original:")
print(sample_text)

print("\nPOS-Aware Lemmatized:")
print(processed_text)

Original:
The children were running quickly and eating better foods while their parents went shopping.

POS-Aware Lemmatized:
child run quickly eat good food parent go shop


# 🔢 Creating Bag of Words with POS-Aware Lemmatization

After preprocessing, the POS-aware lemmatized corpus is converted into numerical features using `CountVectorizer`.

The pipeline is now:

**Raw SMS**

⬇️

**Text Cleaning**

⬇️

**Tokenization**

⬇️

**POS Tagging**

⬇️

**POS-Aware Lemmatization**

⬇️

**CountVectorizer**

⬇️

**Bag of Words Feature Matrix**

The resulting matrix can be used as input for machine learning algorithms such as:

- Naive Bayes
- Logistic Regression
- Support Vector Machine
- Random Forest

The `max_features` parameter limits the vocabulary to a specified number of selected features.

In [34]:
# Create the Bag of Words vectorizer
pos_lemma_vectorizer = CountVectorizer(
    max_features=100
)

# Learn the vocabulary and transform the corpus
X_pos_lemmatized = pos_lemma_vectorizer.fit_transform(
    pos_lemmatized_corpus
)

print(
    "POS-Aware BoW Matrix Shape:",
    X_pos_lemmatized.shape
)

POS-Aware BoW Matrix Shape: (5572, 100)


In [35]:
# Convert only a small sample to a dense array
X_pos_lemmatized[:5].toarray()

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0,

In [36]:
# Display the vocabulary features
pos_lemma_vectorizer.get_feature_names_out()

array(['already', 'amp', 'ask', 'babe', 'back', 'call', 'care', 'cash',
       'claim', 'co', 'come', 'da', 'day', 'dear', 'dont', 'even', 'feel',
       'find', 'free', 'friend', 'get', 'give', 'go', 'good', 'great',
       'gt', 'happy', 'hey', 'hi', 'home', 'hope', 'im', 'keep', 'know',
       'last', 'late', 'later', 'leave', 'let', 'life', 'like', 'lor',
       'love', 'lt', 'make', 'meet', 'message', 'min', 'miss', 'mobile',
       'morning', 'msg', 'much', 'na', 'need', 'new', 'night', 'number',
       'oh', 'ok', 'one', 'phone', 'pick', 'please', 'pls', 'prize',
       'really', 'reply', 'right', 'say', 'see', 'send', 'sleep', 'sorry',
       'still', 'stop', 'take', 'tell', 'text', 'thing', 'think', 'time',
       'today', 'tomorrow', 'tone', 'try', 'txt', 'ur', 'wait', 'wan',
       'want', 'wat', 'way', 'week', 'well', 'win', 'work', 'www', 'yeah',
       'yes'], dtype=object)

In [37]:
# Create a readable DataFrame for the first 5 messages
bow_sample = pd.DataFrame(
    X_pos_lemmatized[:5].toarray(),
    columns=pos_lemma_vectorizer.get_feature_names_out()
)

bow_sample

,already,amp,ask,babe,back,call,care,cash,claim,co,...,want,wat,way,week,well,win,work,www,yeah,yes
0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 📊 Comparing the Three Preprocessing Approaches

We now have three different text normalization strategies before Bag of Words vectorization.

| Approach | Method | Main Advantage | Main Limitation |
|---|---|---|---|
| Snowball Stemming | Rule-based stemming | Fast and simple | May produce non-dictionary stems |
| Basic Lemmatization | WordNet with default POS | Produces meaningful noun lemmas | May fail for verbs and adjectives |
| POS-Aware Lemmatization | POS tagging + WordNet | More linguistically accurate | More computationally expensive |

## Example

Consider:

`The children were running and eating better foods.`

### Snowball Stemming

May produce:

`children run eat better food`

### Basic Lemmatization

May produce:

`child running eating better food`

### POS-Aware Lemmatization

Can produce:

`child run eat good food`

The POS-aware approach can correctly identify:

`children → Noun → child`

`running → Verb → run`

`eating → Verb → eat`

`better → Adjective → good`

`foods → Noun → food`

This demonstrates why POS information can improve lemmatization quality.

However, more linguistically accurate preprocessing does **not automatically guarantee better machine learning accuracy**.

For a classification problem such as spam detection, the best preprocessing approach should be determined experimentally by comparing model performance.

In [38]:
print(
    "Snowball Stemmed BoW:",
    X_stemmed.shape
)

print(
    "Basic Lemmatized BoW:",
    X_lemmatized.shape
)

print(
    "POS-Aware Lemmatized BoW:",
    X_pos_lemmatized.shape
)

Snowball Stemmed BoW: (5572, 100)
Basic Lemmatized BoW: (5572, 100)
POS-Aware Lemmatized BoW: (5572, 100)


In [39]:
print("Snowball Vocabulary:")
print(
    stemmed_vectorizer.get_feature_names_out()[:20]
)


print("\nBasic Lemmatization Vocabulary:")
print(
    lemma_vectorizer.get_feature_names_out()[:20]
)


print("\nPOS-Aware Lemmatization Vocabulary:")
print(
    pos_lemma_vectorizer.get_feature_names_out()[:20]
)

Snowball Vocabulary:
['alreadi' 'amp' 'ask' 'babe' 'back' 'call' 'care' 'cash' 'claim' 'come'
 'da' 'day' 'dear' 'dont' 'even' 'feel' 'find' 'free' 'friend' 'get']

Basic Lemmatization Vocabulary:
['already' 'amp' 'anything' 'ask' 'babe' 'back' 'call' 'care' 'cash'
 'claim' 'co' 'com' 'come' 'da' 'day' 'dear' 'dont' 'find' 'free' 'friend']

POS-Aware Lemmatization Vocabulary:
['already' 'amp' 'ask' 'babe' 'back' 'call' 'care' 'cash' 'claim' 'co'
 'come' 'da' 'day' 'dear' 'dont' 'even' 'feel' 'find' 'free' 'friend']


# 🧠 Key Takeaways — POS-Aware Lemmatized Bag of Words

- `WordNetLemmatizer` works more accurately when the correct POS is provided.
- NLTK's `pos_tag()` returns Penn Treebank POS tags.
- Penn Treebank tags must be mapped to WordNet POS categories.
- WordNet primarily supports **nouns, verbs, adjectives, and adverbs** for lemmatization.
- POS-aware lemmatization can correctly normalize words such as:

  `running → run`

  `went → go`

  `better → good`

  `children → child`

- Unsupported POS categories can be preserved unchanged.
- The processed corpus is converted into numerical features using `CountVectorizer`.
- POS-aware lemmatization is more linguistically accurate than basic lemmatization but requires additional computation.
- Better linguistic normalization does not necessarily guarantee better classification accuracy.
- The best preprocessing technique should be selected by evaluating each approach on the same machine learning model.

---

# 🏆 Final POS-Aware BoW Pipeline

**Raw SMS Messages**

⬇️

**Text Cleaning**

⬇️

**Lowercasing**

⬇️

**Word Tokenization**

⬇️

**Stopword Removal**

⬇️

**POS Tagging**

⬇️

**NLTK POS → WordNet POS Mapping**

⬇️

**POS-Aware Lemmatization**

⬇️

**Processed Corpus**

⬇️

**CountVectorizer**

⬇️

**Bag of Words Matrix**

⬇️

**Machine Learning Model**

## 📌 In Short

`Raw Text → Clean → Tokenize → POS Tag → Lemmatize → Vectorize → Model`